# Análise de Resultados e Escala de Decisão
## Global Solution 2026 — Monitoramento de Riscos Ambientais

Notebook interativo que executa os algoritmos, mede desempenho e apresenta a escala de decisão.

In [6]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
from data_loader import carregar_cenario_rs, carregar_cenario_conectividade
from data_structures import Grafo, BinarySearchTree
from brute_force import mst_forca_bruta
from greedy import mst_prim, cobertura_vizinho_proximo, exibir_mst, priorizar_por_risco
from performance_monitor import benchmark, imprimir_tabela

## 1. Cenário A — Enchentes RS: MST (Prim) vs Força Bruta

In [7]:
v, _g, a = carregar_cenario_rs()
grafo = Grafo.de_dados(v, a)
mst_g, peso_g, cont_g = mst_prim(grafo)
mst_fb, peso_fb, cont_fb = mst_forca_bruta(grafo)
print(exibir_mst(grafo, mst_g, peso_g))
print(f'\nPrim: {peso_g}  |  Forca Bruta: {peso_fb}  |  gap = {round(100*(peso_g-peso_fb)/peso_fb,3)}%')
print(f'FB chamadas recursivas = {cont_fb.chamadas_recursivas}')

MST (Prim) — peso total = 10.5
  Porto Alegre --(0.4)--> Canoas
  Porto Alegre --(0.5)--> Guaíba
  Canoas --(0.5)--> São Leopoldo
  São Leopoldo --(0.3)--> Novo Hamburgo
  Novo Hamburgo --(0.6)--> Montenegro
  Montenegro --(0.4)--> Triunfo
  Montenegro --(1.1)--> Estrela
  Estrela --(0.2)--> Lajeado
  Novo Hamburgo --(1.3)--> Caxias do Sul
  Guaíba --(2.0)--> Pelotas
  Porto Alegre --(3.2)--> Santa Maria

Prim: 10.5  |  Forca Bruta: 10.5  |  gap = 0.0%
FB chamadas recursivas = 5723


## 2. Priorização por risco (via BST)

In [8]:
for idm, nome, risco in priorizar_por_risco(grafo, k=5):
    print(f'{nome}: risco={risco}')

Lajeado: risco=0.93
Triunfo: risco=0.89
Canoas: risco=0.86
Estrela: risco=0.82
Novo Hamburgo: risco=0.8


## 3. Benchmark de escalabilidade e gap de otimalidade

Comparamos dois gulosos contra o ótimo: o **Prim** (avalia toda a fronteira, é ótimo para MST) e o **vizinho-mais-próximo** (decide só pela vizinhança do último nó, é míope/subótimo).

In [9]:
regs = benchmark()
imprimir_tabela(regs)

   N |  FB tempo(ms) |  Prim(ms) | Prim peso |  NN peso |  gap Prim |  gap NN
-----------------------------------------------------------------------------
   5 |         0.112 |    0.0576 |      12.8 |     12.8 |      0.0% |    0.0%
   8 |         1.262 |    0.0358 |     19.03 |    19.03 |      0.0% |    0.0%
  10 |         3.256 |    0.0378 |     21.84 |    21.84 |      0.0% |    0.0%
  12 |        44.837 |    0.0948 |     21.93 |    22.24 |      0.0% |  1.414%
  20 |             — |    0.0687 |     29.21 |    29.21 |         — |       —
  50 |             — |    0.2800 |      86.0 |    95.95 |         — |       —
 100 |             — |    0.3095 |    192.13 |   211.83 |         — |       —


In [10]:
# Gap de otimalidade dos dois gulosos (referência: FB para N<=12; Prim para N>12)
print(f"{'N':>4} | {'Prim ms':>8} | {'peso Prim':>9} | {'peso NN':>8} | {'gap Prim':>9} | {'gap NN':>8}")
print('-'*62)
for r in regs:
    otimo = r['fb_peso'] if r.get('fb_peso') else r['greedy_peso']
    gp = round(100*(r['greedy_peso']-otimo)/otimo, 3)
    gn = round(100*(r['nn_peso']-otimo)/otimo, 3)
    print(f"{r['N']:>4} | {r['greedy_tempo_ms']:>8.3f} | {r['greedy_peso']:>9} | {r['nn_peso']:>8} | {gp:>8}% | {gn:>7}%")

   N |  Prim ms | peso Prim |  peso NN |  gap Prim |   gap NN
--------------------------------------------------------------
   5 |    0.058 |      12.8 |     12.8 |      0.0% |     0.0%
   8 |    0.036 |     19.03 |    19.03 |      0.0% |     0.0%
  10 |    0.038 |     21.84 |    21.84 |      0.0% |     0.0%
  12 |    0.095 |     21.93 |    22.24 |      0.0% |   1.414%
  20 |    0.069 |     29.21 |    29.21 |      0.0% |     0.0%
  50 |    0.280 |      86.0 |    95.95 |      0.0% |   11.57%
 100 |    0.309 |    192.13 |   211.83 |      0.0% |  10.253%


## 4. Escala de decisão (4 níveis)

A escala pondera **qualidade da solução** (gap em relação ao ótimo) contra **custo computacional**. Os gaps abaixo são típicos; os valores exatos do guloso míope variam por instância aleatória (ver tabela da célula anterior).

| Nível | Configuração | Qualidade (gap) | Custo computacional | Recomendação |
|-------|--------------|-----------------|---------------------|--------------|
| **1 — Excelente** | Prim, qualquer N viável | 0% (ótimo p/ MST) | µs a < 1 ms | **Recomendado** |
| **2 — Validação** | Força Bruta, N ≤ 12 | 0% (é o próprio oráculo) | alto já em N=12 (~25 ms) | Só para validar |
| **3 — Aceitável** | Guloso míope, N grande | subótima (~1% a ~12%) | µs | Só se a simplicidade exigir |
| **4 — Inviável** | Força Bruta, N > 12 | n/d (não termina) | explosivo (N^(N-2)) | Descartar |

**Leitura.** O Prim domina: mesma ordem de custo do guloso míope, porém sempre ótimo. A Força Bruta serve apenas como oráculo em N ≤ 12, pois explode acima disso. O guloso míope mostra que *nem todo guloso é ótimo* — a qualidade da decisão local é o que separa a solução exata de uma aproximação de até ~12%.